# Transformers and Attention Mechanisms - Solutions

Worked solutions for the exercise tasks. The lesson code is kept because the exercises build on it, so this notebook runs end to end on its own.

## Setup and Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RSEED = 42
np.random.seed(RSEED)
tf.random.set_seed(RSEED)

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["figure.titlesize"] = 14
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {len(gpus)} GPU(s)")
else:
    print("No GPU found, using CPU")

## Part 1: Understanding the Attention Mechanism from Scratch

Let's start by implementing the basic Attention mechanism to understand how it works.

We build it directly from the formula rather than calling the built-in `layers.MultiHeadAttention`, because the shapes are the part people find confusing and writing them out once makes the rest of the notebook readable. In your own projects use the built-in layer: it is tested and faster.

### Scaled Dot-Product Attention

One detail deserves attention: the division by $\sqrt{d_k}$. Dot products of two random vectors grow with dimension, so without that scaling the scores fed into the softmax become large, the softmax saturates, and the gradients through it shrink towards zero. Dividing by $\sqrt{d_k}$ keeps the scores in a range where the softmax stays responsive.

The optional mask is added before the softmax, multiplied by a large negative number, so masked positions receive effectively zero weight.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):

    d_k = tf.cast(tf.shape(Q)[-1], tf.float32)

    scores = tf.matmul(Q, K, transpose_b=True)

    scores = scores / tf.math.sqrt(d_k)

    if mask is not None:
        scores += mask * -1e9

    attention_weights = tf.nn.softmax(scores, axis=-1)

    output = tf.matmul(attention_weights, V)

    return output, attention_weights


print("Scaled Dot-Product Attention implemented!")

### Attention Visualization

Let's see how Attention works using a simple example.

In [ ]:
seq_len = 5
d_model = 8

Q = tf.random.normal([1, seq_len, d_model])
K = tf.random.normal([1, seq_len, d_model])
V = tf.random.normal([1, seq_len, d_model])

output, attention_weights = scaled_dot_product_attention(Q, K, V)

print(f"Query Shape: {Q.shape}")
print(f"Key Shape: {K.shape}")
print(f"Value Shape: {V.shape}")
print(f"Output Shape: {output.shape}")
print(f"Attention Weights Shape: {attention_weights.shape}")

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    attention_weights[0].numpy(),
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    xticklabels=[f"Token {i}" for i in range(seq_len)],
    yticklabels=[f"Token {i}" for i in range(seq_len)],
)
plt.title("Attention Weights Matrix")
plt.xlabel("Key/Value Position")
plt.ylabel("Query Position")
plt.tight_layout()
plt.show()

## Part 2: Multi-Head Attention

A single attention operation produces one set of weights, so it can only express one notion of relevance at a time. Multi-head attention splits the model dimension into several smaller heads, runs attention in each one independently, then concatenates the results. One head can track syntactic agreement while another tracks subject and object, and the split costs nothing extra: with `d_model` 64 and 8 heads, each head works in 8 dimensions.

```mermaid
flowchart LR
    x[Input sequence] --> q[Project to Q, K, V]
    q --> split[Split into heads]
    split --> attn[Attention in each head]
    attn --> concat[Concatenate heads]
    concat --> out[Final projection]
```

In [ ]:
class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):

        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.depth = d_model // num_heads

        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)

        self.dense = layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        """Splits the last dimension into (num_heads, depth)"""
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, query, key, value, mask=None):
        batch_size = tf.shape(query)[0]

        q = self.wq(query)
        k = self.wk(key)
        v = self.wv(value)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        scaled_attention, attention_weights = scaled_dot_product_attention(
            q, k, v, mask
        )

        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))

        output = self.dense(concat_attention)

        return output, attention_weights


print("Multi-Head Attention implemented!")

In [ ]:
d_model = 64
num_heads = 8
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)

x = tf.random.normal([2, seq_len, d_model])

output, attn_weights = mha(x, x, x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

## Part 3: Positional Encoding

Since Transformers have no recurrent structure, we must explicitly add position information.

This matters more than it first appears. Attention is a weighted sum, and a sum does not care about order: shuffle the input positions and the output is shuffled identically, with the same values. Without position information a Transformer cannot tell "the dog bit the man" from "the man bit the dog".

The sinusoidal scheme used here adds a fixed pattern of sines and cosines at different frequencies, so each position gets a distinct signature and nearby positions get similar ones. The alternative is a learned positional embedding, which most modern models use; the fixed version needs no parameters and extends to sequences longer than any seen in training.

In [ ]:
def get_positional_encoding(seq_len, d_model):

    positions = np.arange(seq_len)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates

    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]

    return tf.cast(pos_encoding, dtype=tf.float32)

In [ ]:
pos_encoding = get_positional_encoding(100, 128)

plt.figure(figsize=(12, 6))
plt.pcolormesh(pos_encoding[0], cmap="RdBu")
plt.xlabel("Embedding Dimension")
plt.ylabel("Position")
plt.colorbar()
plt.title("Positional Encoding Visualisierung")
plt.tight_layout()
plt.show()

## Part 4: Feed-Forward Network

Attention moves information between positions but applies no non-linear transformation of its own beyond the softmax. The feed-forward block supplies that: the same two-layer network is applied to each position separately, expanding to a larger inner dimension (`dff`, here four times `d_model`) and projecting back. This is where most of a Transformer's parameters live.

In [ ]:
def point_wise_feed_forward_network(d_model, dff):

    return keras.Sequential(
        [layers.Dense(dff, activation="relu"), layers.Dense(d_model)]
    )


ffn = point_wise_feed_forward_network(512, 2048)
x = tf.random.normal([2, 10, 512])
output = ffn(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

## Part 5: Encoder Layer

An encoder layer wires the two pieces together with a residual connection and layer normalisation around each:

```mermaid
flowchart TD
    inp[Input] --> att[Multi-head attention]
    att --> add1[Add input back, then normalise]
    inp --> add1
    add1 --> ff[Feed-forward network]
    ff --> add2[Add input back, then normalise]
    add1 --> add2
    add2 --> outp[Output]
```

Both additions matter. The residual path gives the gradient a route that skips the sublayer entirely, which is what makes deep stacks trainable, and it is the same idea as the LSTM cell state in the previous notebook. Layer normalisation keeps activations at a consistent scale across positions, which is preferred over batch normalisation here because sequence batches vary in length and content.

In [ ]:
class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):

        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_feed_forward_network(d_model, dff)

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, training, mask=None):

        attn_output, _ = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2


encoder_layer = EncoderLayer(d_model=512, num_heads=8, dff=2048)
x = tf.random.normal([2, 10, 512])
output = encoder_layer(x, training=False)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

## Part 6: Complete Transformer Encoder

Stacking encoder layers is what gives a Transformer depth. One layer lets every position consult every other once; a second layer lets it consult positions that have themselves already gathered context, so the representation becomes progressively more contextual. The original paper stacks 6 layers, BERT base uses 12, and large language models use many more.

The positional encoding is added once, at the bottom of the stack, before the first layer.

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(
        self,
        num_layers,
        d_model,
        num_heads,
        dff,
        maximum_position_encoding,
        dropout_rate=0.1,
    ):

        super().__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.pos_encoding = get_positional_encoding(maximum_position_encoding, d_model)

        self.enc_layers = [
            EncoderLayer(d_model, num_heads, dff, dropout_rate)
            for _ in range(num_layers)
        ]

        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, training=None, mask=None):
        seq_len = tf.shape(x)[1]

        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)

        for enc_layer in self.enc_layers:
            x = enc_layer(x, training=training, mask=mask)

        return x


transformer_encoder = TransformerEncoder(
    num_layers=4,
    d_model=128,
    num_heads=8,
    dff=512,
    maximum_position_encoding=100,
    dropout_rate=0.1,
)

x = tf.random.normal([2, 20, 128])
output = transformer_encoder(x, training=False)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

## Part 7: Practical Application - Sequence Classification

Now let's build a complete model for sequence classification.

To turn the encoder into a classifier we need to reduce a sequence of vectors to a single vector. Here we average over positions with `GlobalAveragePooling1D`, which is simple and order-independent. The common alternative is to prepend a dedicated classification token and read its final representation, as BERT does, which lets the model learn how to pool rather than fixing the rule in advance.

Keep that choice in mind when reading the results below: averaging is exactly the operation that makes the synthetic task here so easy.

In [ ]:
class TransformerClassifier(keras.Model):
    def __init__(
        self,
        num_layers,
        d_model,
        num_heads,
        dff,
        maximum_position_encoding,
        num_classes,
        dropout_rate=0.1,
    ):

        super().__init__()

        self.encoder = TransformerEncoder(
            num_layers, d_model, num_heads, dff, maximum_position_encoding, dropout_rate
        )

        self.global_avg_pooling = layers.GlobalAveragePooling1D()
        self.dropout = layers.Dropout(dropout_rate)
        self.classifier = layers.Dense(num_classes, activation="softmax")

    def call(self, x, training=False):

        x = self.encoder(x, training=training)

        x = self.global_avg_pooling(x)
        x = self.dropout(x, training=training)

        return self.classifier(x)


print("Transformer Classifier implemented!")

### Example: Sentiment Classification on Synthetic Data

The data below is deliberately simple so the model trains in seconds. Class 0 is drawn from a normal distribution shifted to -1.0 and class 1 from one shifted to +1.0, across all 20 positions and all 64 features.

Be aware of what that means before reading the results. The signal sits in every position equally, so averaging the sequence already separates the two classes perfectly, and no model needs to look at the order of the tokens. It is a useful smoke test that the implementation runs, not a demonstration that attention helps.

In [ ]:
def generate_sentiment_data(num_samples=1000, seq_len=20, d_model=64):
    """
    Generates synthetic sentiment data
    Class 0: Average negative
    Class 1: Average positive
    """
    X = []
    y = []

    for _ in range(num_samples):
        label = np.random.choice([0, 1])

        if label == 0:
            sequence = np.random.randn(seq_len, d_model) - 1.0
        else:
            sequence = np.random.randn(seq_len, d_model) + 1.0

        X.append(sequence)
        y.append(label)

    X = np.array(X).astype("float32")
    y = np.array(y)

    return X, y


X_train, y_train = generate_sentiment_data(num_samples=800, seq_len=20, d_model=64)
X_test, y_test = generate_sentiment_data(num_samples=200, seq_len=20, d_model=64)

print(f"Training data: {X_train.shape}, {y_train.shape}")
print(f"Test data: {X_test.shape}, {y_test.shape}")

In [ ]:
model = TransformerClassifier(
    num_layers=2,
    d_model=64,
    num_heads=4,
    dff=128,
    maximum_position_encoding=50,
    num_classes=2,
    dropout_rate=0.1,
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=20,
    validation_data=(X_test, y_test),
    verbose=1,
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss über Epochen")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy über Epochen")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 8: Comparison Transformer versus LSTM

We now train an LSTM on the same data, so the two architectures can be compared directly.

One caveat first, so the numbers are read correctly. As noted above, this task is separable by averaging alone, so both models reach perfect accuracy almost immediately and accuracy cannot rank them. What the comparison can still show honestly is the cost of each architecture and how each one processes a sequence.

In [ ]:
# Check how separable the task is before comparing models on it.
# GlobalAveragePooling1D reduces each sample to the mean of its 20 x 64 values.

pooled = X_train.mean(axis=(1, 2))

print(
    f"Class 0 pooled mean: {pooled[y_train == 0].mean():.4f} "
    f"(std {pooled[y_train == 0].std():.4f})"
)
print(
    f"Class 1 pooled mean: {pooled[y_train == 1].mean():.4f} "
    f"(std {pooled[y_train == 1].std():.4f})"
)

baseline = ((pooled > 0).astype(int) == y_train).mean()
print(
    f"\nAccuracy of the rule 'predict class 1 if the pooled mean is positive': {baseline:.4f}"
)

The two classes sit about one unit apart with a standard deviation of roughly 0.03, so they are separated by more than thirty standard deviations and do not overlap at all. A single average, with no model and no training, already classifies the data perfectly.

Keep that baseline in mind: any accuracy a model reports on this task is matching a one-line rule, not demonstrating that it learned anything about sequence structure.

In [ ]:
lstm_model = keras.Sequential(
    [
        layers.Input(shape=(20, 64)),
        layers.LSTM(64),
        layers.Dropout(0.1),
        layers.Dense(2, activation="softmax"),
    ]
)

lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("LSTM Model:")
print(lstm_model.summary())

In [ ]:
history_lstm = lstm_model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=20,
    validation_data=(X_test, y_test),
    verbose=1,
)

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["val_loss"], label="Transformer", linewidth=2)
plt.plot(history_lstm.history["val_loss"], label="LSTM", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss: Transformer vs LSTM")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history["val_accuracy"], label="Transformer", linewidth=2)
plt.plot(history_lstm.history["val_accuracy"], label="LSTM", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy: Transformer vs LSTM")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Final metrics:")
print(
    f"Transformer - val loss: {history.history['val_loss'][-1]:.6f}, "
    f"val accuracy: {history.history['val_accuracy'][-1]:.4f}"
)
print(
    f"LSTM        - val loss: {history_lstm.history['val_loss'][-1]:.6f}, "
    f"val accuracy: {history_lstm.history['val_accuracy'][-1]:.4f}"
)

print("\nParameter count:")
print(f"Transformer: {model.count_params():>6}")
print(f"LSTM:        {lstm_model.count_params():>6}")

Both models reach a validation accuracy of 1.0000, which is exactly the baseline computed above, so on this data the comparison is a tie and tells us nothing about which architecture is stronger.

The difference that is real is cost and structure. The Transformer uses 67,074 parameters against the LSTM's 33,154, roughly twice as many, because each encoder layer carries four projection matrices for attention plus a feed-forward block. In exchange it processes all 20 positions in one parallel operation, whereas the LSTM must step through them in order, which is what makes Transformers scale on long sequences and on modern hardware.

To see attention earn its cost you need a task where order matters, for example one where the label depends on which token appears first or on the relationship between two distant positions. Averaging destroys that information, so a pooling-based shortcut can no longer solve it. That is the natural next exercise for this notebook.

## Exercise Solutions

### Exercise 1: Visualise the Attention Weights

Extend the code to visualise the attention weights of a trained model.

In [ ]:
# The encoder layer discards the attention weights it computes, so we call the
# trained multi-head attention directly on the same input the layer would receive:
# the sample plus its positional encoding.

sample = X_test[:1]

encoder = model.encoder
encoded_input = sample + encoder.pos_encoding[:, : sample.shape[1], :]
first_layer = encoder.enc_layers[0]

_, trained_weights = first_layer.mha(encoded_input, encoded_input, encoded_input)

print(f"Attention weights shape: {trained_weights.shape}")
print("(batch, heads, query positions, key positions)")

In [ ]:
num_heads_trained = trained_weights.shape[1]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for head, ax in enumerate(axes.ravel()):
    sns.heatmap(
        trained_weights[0, head].numpy(), cmap="YlOrRd", ax=ax, cbar=False, square=True
    )
    ax.set_title(f"Head {head + 1}")
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")

plt.suptitle("Attention weights per head, first encoder layer (trained model)")
plt.tight_layout()
plt.show()

**Explanation**

The encoder layer computes attention weights internally but returns only the transformed sequence, so the weights are not reachable from the model output. Rather than modify the lesson class, we call the trained layer.mha directly on the input it would have received, which is the sample plus its positional encoding. The result has one matrix per head, and plotting them side by side is the standard way to inspect what a trained model attends to. On this synthetic task the signal is spread evenly across positions, so expect fairly uniform weights rather than sharp structure.


### Exercise 2: Hyperparameters

Experiment with different values for `num_layers`, `num_heads` and `d_model`.

In [ ]:
# Note one constraint before sweeping: the positional encoding is added straight to
# the input, so the input width must equal d_model. Changing d_model therefore means
# regenerating the data at that width, which the lesson's own generator supports.
# Accuracy saturates on this task, so the columns worth reading are parameters and time.

import time

configs = [
    {"num_layers": 1, "num_heads": 2, "d_model": 32},
    {"num_layers": 2, "num_heads": 4, "d_model": 64},
    {"num_layers": 4, "num_heads": 8, "d_model": 64},
]

sweep_results = []

for cfg in configs:
    X_cfg_train, y_cfg_train = generate_sentiment_data(
        num_samples=800, seq_len=20, d_model=cfg["d_model"]
    )
    X_cfg_test, y_cfg_test = generate_sentiment_data(
        num_samples=200, seq_len=20, d_model=cfg["d_model"]
    )

    model_cfg = TransformerClassifier(
        dff=cfg["d_model"] * 2,
        maximum_position_encoding=50,
        num_classes=2,
        dropout_rate=0.1,
        **cfg,
    )
    model_cfg.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    start = time.time()
    history_cfg = model_cfg.fit(
        X_cfg_train,
        y_cfg_train,
        batch_size=32,
        epochs=5,
        validation_data=(X_cfg_test, y_cfg_test),
        verbose=0,
    )
    elapsed = time.time() - start

    sweep_results.append(
        {
            **cfg,
            "params": model_cfg.count_params(),
            "val_accuracy": round(float(history_cfg.history["val_accuracy"][-1]), 4),
            "seconds": round(elapsed, 1),
        }
    )

print(
    f"{'layers':>7} {'heads':>6} {'d_model':>8} {'params':>9} {'val acc':>9} {'seconds':>8}"
)
for r in sweep_results:
    print(
        f"{r['num_layers']:>7} {r['num_heads']:>6} {r['d_model']:>8} "
        f"{r['params']:>9} {r['val_accuracy']:>9} {r['seconds']:>8}"
    )

**Explanation**

The sweep has one structural constraint worth understanding: the positional encoding is added directly to the input, so the input width must equal d_model, which is why the data is regenerated for each width. Accuracy is saturated on this task and therefore useless for ranking, so the informative columns are parameters and seconds. Note that depth and width cost far more than head count: splitting the same d_model into more heads changes how attention is grouped without adding parameters.


### Exercise 3: Real Data

Load a text dataset and train the Transformer on it.

We use the IMDB film review dataset that ships with Keras, so no manual download is needed. Real text needs one component the synthetic example did not: an `Embedding` layer that turns integer token ids into the dense vectors the encoder expects.

In [ ]:
from tensorflow.keras.datasets import imdb

VOCAB_SIZE = 10000  # keep only the most frequent words
MAX_LENGTH = 200  # truncate or pad every review to this length
TRAIN_SAMPLES = 10000

(x_train_raw, y_train_imdb), (x_test_raw, y_test_imdb) = imdb.load_data(
    num_words=VOCAB_SIZE
)

# Reviews vary in length, so pad them to a fixed window the encoder can batch.
x_train_imdb = keras.utils.pad_sequences(x_train_raw, maxlen=MAX_LENGTH)
x_test_imdb = keras.utils.pad_sequences(x_test_raw, maxlen=MAX_LENGTH)

# A subset keeps the training time reasonable on a CPU.
x_train_imdb, y_train_imdb = x_train_imdb[:TRAIN_SAMPLES], y_train_imdb[:TRAIN_SAMPLES]
x_test_imdb, y_test_imdb = x_test_imdb[:2500], y_test_imdb[:2500]

print(f"Training reviews: {x_train_imdb.shape}, labels: {y_train_imdb.shape}")
print(f"Test reviews:     {x_test_imdb.shape}")
print(f"Positive share in training data: {y_train_imdb.mean():.2f}")

In [ ]:
class TextTransformerClassifier(keras.Model):
    def __init__(
        self,
        vocab_size,
        num_layers,
        d_model,
        num_heads,
        dff,
        maximum_position_encoding,
        num_classes,
        dropout_rate=0.1,
    ):
        super().__init__()

        # The embedding is the only new piece: it maps token ids to d_model vectors,
        # which is exactly the input the encoder from Part 6 already expects.
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.encoder = TransformerEncoder(
            num_layers, d_model, num_heads, dff, maximum_position_encoding, dropout_rate
        )
        self.pooling = layers.GlobalAveragePooling1D()
        self.dropout = layers.Dropout(dropout_rate)
        self.classifier = layers.Dense(num_classes, activation="softmax")

    def call(self, x, training=False):
        x = self.embedding(x)
        x = self.encoder(x, training=training)
        x = self.pooling(x)
        x = self.dropout(x, training=training)
        return self.classifier(x)


text_model = TextTransformerClassifier(
    vocab_size=VOCAB_SIZE,
    num_layers=2,
    d_model=64,
    num_heads=4,
    dff=128,
    maximum_position_encoding=MAX_LENGTH,
    num_classes=2,
    dropout_rate=0.1,
)

text_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_text = text_model.fit(
    x_train_imdb,
    y_train_imdb,
    batch_size=64,
    epochs=3,
    validation_data=(x_test_imdb, y_test_imdb),
    verbose=1,
)

print(f"\nFinal validation accuracy: {history_text.history['val_accuracy'][-1]:.4f}")
print(
    f"Majority-class baseline:   {max(y_test_imdb.mean(), 1 - y_test_imdb.mean()):.4f}"
)

**Explanation**

Text needs one component the synthetic example did not: an Embedding layer mapping token ids to d_model vectors, which is exactly what the encoder already expects. The rest of the architecture is reused unchanged, which is the point of the exercise. Two details matter for correctness: maximum_position_encoding must be at least the padded length, and the padding itself is left unmasked here for simplicity. A production model would pass a padding mask so attention ignores the padded positions, and would gain accuracy from doing so.


### Exercise 4: Decoder

Implement a Transformer decoder for sequence-to-sequence tasks.

In [ ]:
def create_look_ahead_mask(size):
    # 1 marks a position the query is not allowed to see. band_part keeps the lower
    # triangle (the past and the present), so the complement hides the future.
    return 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)


print("Look-ahead mask for a sequence of 5 tokens:")
print(create_look_ahead_mask(5).numpy())

In [ ]:
class DecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()

        # Two attention blocks: one over the output built so far, one over the encoder.
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_feed_forward_network(d_model, dff)

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.dropout3 = layers.Dropout(dropout_rate)

    def call(self, x, encoder_output, training=False, look_ahead_mask=None):
        # Masked self-attention: each position may only look at itself and earlier ones.
        attn1, self_weights = self.self_attention(x, x, x, look_ahead_mask)
        out1 = self.layernorm1(x + self.dropout1(attn1, training=training))

        # Cross-attention: queries come from the decoder, keys and values from the encoder.
        attn2, cross_weights = self.cross_attention(
            out1, encoder_output, encoder_output
        )
        out2 = self.layernorm2(out1 + self.dropout2(attn2, training=training))

        ffn_output = self.ffn(out2)
        out3 = self.layernorm3(out2 + self.dropout3(ffn_output, training=training))

        return out3, self_weights, cross_weights


decoder_layer = DecoderLayer(d_model=64, num_heads=4, dff=128)

target_sequence = tf.random.normal([2, 12, 64])  # what the decoder has produced so far
encoder_memory = tf.random.normal([2, 20, 64])  # what the encoder read

decoder_output, self_w, cross_w = decoder_layer(
    target_sequence,
    encoder_memory,
    training=False,
    look_ahead_mask=create_look_ahead_mask(12),
)

print(f"Decoder output shape:          {decoder_output.shape}")
print(f"Masked self-attention weights: {self_w.shape}")
print(
    f"Cross-attention weights:       {cross_w.shape}  (12 target positions over 20 source positions)"
)

**Explanation**

The decoder adds two things to the encoder layer. Masked self-attention lets each output position attend only to itself and to earlier positions, which is what makes generation causal, and cross-attention lets the decoder query the encoder output, which is how information crosses from source to target. Note the asymmetry in the printed shapes: self-attention is 12 by 12 over the target, while cross-attention is 12 by 20, one row per target position over all source positions.


In [ ]:
# Confirm the mask works: every position must put zero weight on later positions.

upper_triangle = np.triu(np.ones((12, 12)), k=1).astype(bool)
future_weight = self_w[0, 0].numpy()[upper_triangle].max()

print(f"Largest attention weight on a future position: {future_weight:.2e}")
print("Each row still sums to 1:", np.allclose(self_w[0, 0].numpy().sum(axis=-1), 1.0))

**Explanation**

This is the check that the mask actually works rather than merely being passed in. Adding the mask multiplied by a large negative number before the softmax drives those weights to zero, so the largest weight on any future position should be zero, and every row should still sum to 1. Verifying a mask this way is worth the two lines: a silently ineffective look-ahead mask lets the model read the answer during training and produces excellent validation scores that collapse at generation time.
